# Part 5: Agent Observability and Evaluation in LearnOS (20 points)

In earlier parts you evaluated agents offline against labelled answers. LearnOS has **no labels and no reward**:
your agent is a study assistant on a simulated student's computer, and whether it "succeeded" is something you
have to define.

That makes observability more interesting, because every run now leaves **two traces**:

| | Agent-side trace (Langfuse) | Env-side trace (`env.env_trace`) |
|---|---|---|
| Records | model calls, tool calls, tokens, latency | what each action did to the workspace and the student |
| Answers | *What did the agent do and why?* | *What happened to the student?* |
| Includes | the agent's view of the student activity stream (`[activity] ...` lines in tool outputs) | the same stream, plus the student's hidden state once the grader reveals it |

The two are linked by an `episode_id`, which appears in Langfuse as trace metadata.

A failure category to watch for alongside reasoning / tool output / prompt / infrastructure is
**observation**: the agent reasoned correctly from what it saw, but what it saw was misleading. For example,
the activity tracker only sees the screen, so a student on their phone looks "idle".

### Setup: connect to LearnOS

Start the environment first (`docker compose up` in the `learnos/` folder). The UI at
http://localhost:8080 shows each run live.

In [ ]:
%pip install -q langfuse 'smolagents[telemetry]' openinference-instrumentation-smolagents pandas
%pip install -q -e ../client

In [ ]:
import os, json
from pathlib import Path
from learnos_client import LearnOS, make_tools, run_episode, run_many, compare, trace_table

env = LearnOS(os.getenv("LEARNOS_URL", "http://localhost:8000"),
              grader_token=os.getenv("LEARNOS_GRADER_TOKEN", "change-me"))   # the agent never gets this token
print(env.health())

INSTANCES = {p.stem: json.loads(p.read_text()) for p in Path("../instances").glob("*.json")}
print("instances:", list(INSTANCES))
print(len(env.tools()), "tools:", [t["name"] for t in env.tools()])

#### Problem 1: Setting up an Observability Monitor (5 points)

Set up Langfuse (or any OpenTelemetry-compatible backend) so every LearnOS run is traceable end to end.
**Instrument before you create any agent.**

In [ ]:
from getpass import getpass

# Use environment variables only; do not hardcode secrets in notebook source.
if not os.getenv("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("Enter LANGFUSE_SECRET_KEY: ")
if not os.getenv("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("Enter LANGFUSE_PUBLIC_KEY: ")
os.environ.setdefault("LANGFUSE_BASE_URL", "https://us.cloud.langfuse.com")

from langfuse import get_client
langfuse = get_client()
print("Langfuse ready" if langfuse.auth_check() else "Authentication failed: check keys and host.")

from openinference.instrumentation.smolagents import SmolagentsInstrumentor
SmolagentsInstrumentor().instrument()

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
from smolagents import OpenAIServerModel, ToolCallingAgent

model = OpenAIServerModel(model_id="gpt-5.4-mini")   # or your TransformersModel from Part 3

def make_agent(env, include=None, max_steps=20):
    """An agent factory: run_episode() calls this once per episode.
    include=[...] restricts the tool list (a tool-set-size experiment)."""
    return ToolCallingAgent(tools=make_tools(env, include=include), model=model,
                            max_steps=max_steps, verbosity_level=1)
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

#### Problem 2: Record and Inspect Traces (5 points)

Run your agent and inspect the runs in Langfuse **and** in the env trace.

Minimum requirements:
- Show evidence that at least **5 runs** were recorded as traces (screenshot of the Langfuse trace list).
- For at least **2 runs**, identify the model call count, the tool call sequence, and where most latency occurred.
- Include at least **1 run where behaviour was incorrect or suboptimal**, and diagnose whether the issue came from
  **reasoning, tool output, prompt/instructions, infrastructure, or observation** (the agent's view of the student
  was wrong or incomplete).
- Link each diagnosis to specific evidence in **both** traces: a Langfuse span (name, timing, output) and the
  matching env-trace row (step, activity, and hidden state where revealed).

Each run's Langfuse trace is named `<instance>#<seed>`, its session is the config name, and its metadata holds
`episode_id`. Filter by `episode_id` in Langfuse to find the trace that matches an env trace.

In [ ]:
import pandas as pd
inst = INSTANCES["friday-build-01"]

# Five runs, each with a different simulated student (seed, seed+1, ...).
# Success is a placeholder here; you define it properly in Problem 3.
baseline = run_many(env, [inst], make_agent, success=lambda g: True, k=5, config="baseline")

runs = baseline["rows"][0]["runs"]
pd.DataFrame([{k: r.get(k) for k in ("episode_id", "seed", "model_calls", "tool_calls", "slowest_step",
                                      "input_tokens", "output_tokens", "wall_s", "agent_error")} for r in runs])

In [ ]:
# Tool call sequence for two runs
for r in runs[:2]:
    print(r["episode_id"], "\n  ", " -> ".join(r["tool_sequence"]), "\n")

In [ ]:
# The env side of one run. Level-1 episodes reveal hidden state once they have ended,
# so the last columns show what was actually happening to the student at each step.
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
EPISODE = runs[0]["episode_id"]
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================
trace_table(env.env_trace(EPISODE))

**Diagnosis template** (one per suboptimal run):

- **Run / episode_id:**
- **What went wrong (behaviour):**
- **Category:** reasoning | tool output | prompt/instructions | infrastructure | observation
- **Langfuse evidence:** span name, step number, timing, or output text
- **Env-trace evidence:** step, activity, hidden-state change
- **Would a different signal have caught it?** e.g. quiz score vs `post_test`, tracked feed minutes vs true feed minutes

#### Problem 3: Online Evaluation (10 points)

Compare agent configurations on the **same students** (same instances and seeds).

First write your success rule. LearnOS gives you **signals, never a reward**: `g(signal)` reads them through the
grader. Tag each signal you use as **direct** (`post_test`, `mastery_delta`, `true_state`) or **proxy**
(`proxies`: quiz mean, tracked minutes per app, nudges, ...). Direct signals are unavailable at level 2 and in
live mode, and `true_state` at level 1 only becomes available once the episode ends.

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
def success(g):
    """DIRECT: the student passes a grader-run delayed exam (agent-independent, fixed items)."""
    return g("post_test", delay_hours=48)["post_test"] > 0.5

def reward(g):
    """A PROXY you might be tempted by. What does it miss, and how could an agent game it?"""
    p = g("proxies")
    return p["quiz_mean"] or 0.0
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

Minimum requirements:
- Change **at least two parameters**. LearnOS-specific options alongside the model and `max_steps`:
  - **tool-set size:** `make_tools(env, include=[...])`. For example, remove `feed_scroll`, or remove `quiz_run` (no sensor at all)
  - **observability level:** set `"level"` to 0 (sees the whole workspace), 1 (sees only what it opens), or 2 (the world changes mid-run)
  - **system instructions:** e.g. "never give answers directly"
- Evaluate each configuration on the **same ≥5 seeds** (`run_many(..., k=5)`).
- Track **at least 3 metrics** per configuration: one correctness (your `success`), one process (model calls,
  tool calls, error rate), and one operational (latency, tokens).
- Attach screenshots of the Langfuse sessions view (each config is one session).

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================
CONFIGS = {
    "baseline":      dict(factory=make_agent, instance=inst),
    "no-quiz-tool":  dict(factory=lambda e: make_agent(e, include=[t["name"] for t in e.tools() if t["name"] != "quiz_run"]),
                          instance=inst),
    "full-view-L0":  dict(factory=make_agent, instance={**inst, "level": 0}),
}
# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

results = [run_many(env, [c["instance"]], c["factory"], success, k=5, reward=reward, config=name)
           for name, c in CONFIGS.items()]
table = compare(results)
table

In [ ]:
# Proxy vs direct across all runs: does the reward you wrote agree with the outcome?
rows = [r for res in results for row in res["rows"] for r in row["runs"]]
pd.DataFrame([{"config": r["config"], "seed": r["seed"], "reward(proxy)": r["reward"], "success(direct)": r["ok"]}
              for r in rows]).sort_values("reward(proxy)", ascending=False)

**Deliverables**
- One comparison table with every configuration and all reported metrics (the `table` above, plus Langfuse cost if you use it).
- A short discussion (6–8 sentences): how your parameter changes affected results, where trade-offs appeared, and
  which setup you would deploy. Did your **proxy** reward rank configurations the same way as your **direct**
  success rule? If not, which runs disagreed, and what do their two traces show?
- Online evaluation going forward: an **LLM-as-a-judge** reading a Langfuse trace is another proxy. How would you
  check it against `post_test`, and what would you use in live mode, where `post_test` is unavailable?